Imports and Setup

In [ ]:
import cv2
import numpy as np
import os
from modules.calibration import calibrate_camera_tsai, get_corrected_camera_center
from modules.visualization import draw_3d_cube, show_result

# Configuration
CHECKER = (9, 6)
IMG_PATH = 'data/Mycheckerboarde.png'

Image Processing and Corner Detection

In [ ]:
img = cv2.imread(IMG_PATH)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, corners = cv2.findChessboardCorners(gray, CHECKER, None)

if ret:
    # Pixel points (homogeneous)
    p = np.vstack([corners.squeeze().T, np.ones((1, corners.shape[0]))])
    
    # World points (homogeneous)
    grid = np.indices(CHECKER).T.reshape(-1, 2)
    P = np.vstack([grid.T, np.zeros((1, grid.shape[0])), np.ones((1, grid.shape[0]))])
    
    # Visualize detected corners
    img_corners = cv2.drawChessboardCorners(img.copy(), CHECKER, corners, ret)
    show_result(img_corners, title="Detected Checkerboard Corners")
else:
    print("Error: Checkerboard not found.")

Calibration and 3D Projection

In [ ]:
# Execute Tsai Calibration
K, R, T, M = calibrate_camera_tsai(p, P)
C = get_corrected_camera_center(R, T)

print(f"Intrinsic Matrix (K):\n{K}")
print(f"\nCamera Center (C):\n{C.flatten()}")

# Define 3D Cube vertices (from your assignment data)
vertices = np.array([
    [2, 3, 3, 2, 2, 3, 3, 2],
    [1, 1, 2, 2, 1, 1, 2, 2],
    [0, 0, 0, 0, -1, -1, -1, -1],
    [1, 1, 1, 1, 1, 1, 1, 1]
])

# Project 3D -> 2D
proj = M @ vertices
proj /= proj[2, :] # Perspective divide
pts_2d = proj[:2, :].T.astype(int)

# Render and Display
final_img = draw_3d_cube(img, pts_2d)
show_result(final_img, title="Final 3D Reconstruction Overlay")